# Linear Models from Scratch

Exercise linear regression, logistic regression, and perceptron with reusable OOP implementations and held-out checks.

- **Study time:** 45-60 minutes
- **Prerequisites:** NumPy broadcasting, gradients, and the estimator contract
- **Mode:** `quick`
- **Data policy:** no external files or downloads; seeded synthetic train/test splits only
- **Provenance:** rebuilt from the legacy NumPy ML-from-scratch notebook; rough cells removed and evaluation corrected

Output convention: every retained textual result begins with a label that identifies the operation that produced it.
Annotation convention: comments explain intent, shape changes, invariants, or subtle API behavior; obvious Python syntax is left uncommented.


In [1]:
import sys
from pathlib import Path


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").exists():
            return candidate  # Anchor imports to the repository, not the launch directory.
    raise RuntimeError("Run this notebook from inside the DataCoding project")


PROJECT_ROOT = find_project_root()
source_dir = str(PROJECT_ROOT / "src")
if source_dir not in sys.path:
    sys.path.insert(0, source_dir)  # Prefer this checkout's reusable implementations.

In [2]:
import numpy as np

from datacoding.algorithms import LinearRegressionGD, LogisticRegressionGD, PerceptronClassifier

rng = np.random.default_rng(21)


def show(label, value):
    print(f"\n--- {label} ---\n{value}")


def split(X, y, test_fraction=0.25):
    indices = rng.permutation(len(X))  # Apply one permutation to keep X and y aligned.
    cut = int(len(X) * (1 - test_fraction))
    train_index, test_index = indices[:cut], indices[cut:]
    return X[train_index], X[test_index], y[train_index], y[test_index]


def standardize_train_test(X_train, X_test):
    mean = X_train.mean(axis=0)  # Learn preprocessing from training rows only.
    scale = X_train.std(axis=0) + 1e-12
    return (X_train - mean) / scale, (X_test - mean) / scale

## 1. Linear regression: gradient descent with a least-squares reference


In [3]:
X = rng.normal(size=(500, 3))
true_coef = np.array([2.0, -3.0, 0.5])
y = (
    X @ true_coef + 1.2 + rng.normal(scale=0.3, size=len(X))
)  # Signal + intercept + irreducible noise.
X_train, X_test, y_train, y_test = split(X, y)
X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

linear = LinearRegressionGD(learning_rate=0.08, max_iter=3_000, l2=0.0001)
linear.fit(X_train_scaled, y_train)
test_prediction = linear.predict(X_test_scaled)
test_mse = np.mean((test_prediction - y_test) ** 2)

train_design = np.column_stack(
    [X_train_scaled, np.ones(len(X_train_scaled))]
)  # Append intercept column.
test_design = np.column_stack([X_test_scaled, np.ones(len(X_test_scaled))])
least_squares_parameters = np.linalg.lstsq(train_design, y_train, rcond=None)[
    0
]  # Stable reference solution without an inverse.
least_squares_prediction = test_design @ least_squares_parameters
least_squares_mse = np.mean((least_squares_prediction - y_test) ** 2)

show("Linear regression | train/test shapes", (X_train_scaled.shape, X_test_scaled.shape))
show("Linear regression | learned coefficients in scaled space", linear.coef_)
show("Linear regression | held-out MSE", test_mse)
show("Linear regression | least-squares held-out MSE", least_squares_mse)
show(
    "Linear regression | first and final objective",
    (linear.loss_history_[0], linear.loss_history_[-1]),
)


--- Linear regression | train/test shapes ---
((375, 3), (125, 3))

--- Linear regression | learned coefficients in scaled space ---
[ 1.91533401 -2.91128771  0.44230695]

--- Linear regression | held-out MSE ---
0.07680495095463964

--- Linear regression | least-squares held-out MSE ---
0.07679216958970347

--- Linear regression | first and final objective ---
(13.683709237565681, 0.08914930652984013)


## 2. Logistic regression: probability plus threshold


In [4]:
negative = rng.normal(loc=[-1.5, -1.0], scale=0.9, size=(300, 2))
positive = rng.normal(loc=[1.5, 1.0], scale=0.9, size=(300, 2))
X = np.vstack([negative, positive])
y = np.array([0] * len(negative) + [1] * len(positive))
X_train, X_test, y_train, y_test = split(X, y)
X_train_scaled, X_test_scaled = standardize_train_test(X_train, X_test)

logistic = LogisticRegressionGD(learning_rate=0.2, max_iter=3_000, l2=0.001)
logistic.fit(X_train_scaled, y_train)
probabilities = logistic.predict_proba(X_test_scaled)[:, 1]  # Positive-class probability.
prediction = logistic.predict(X_test_scaled)
accuracy = np.mean(prediction == y_test)
log_loss = -np.mean(
    y_test * np.log(np.clip(probabilities, 1e-12, 1.0))  # Clip only for numerical safety in log.
    + (1 - y_test) * np.log(np.clip(1 - probabilities, 1e-12, 1.0))
)

show("Logistic regression | probability range", (probabilities.min(), probabilities.max()))
show("Logistic regression | held-out accuracy", accuracy)
show("Logistic regression | held-out log loss", log_loss)
show(
    "Logistic regression | first five probability/label pairs",
    np.column_stack([probabilities[:5], y_test[:5]]),
)


--- Logistic regression | probability range ---
(np.float64(1.5142348662448846e-05), np.float64(0.9999315842382075))

--- Logistic regression | held-out accuracy ---
0.96

--- Logistic regression | held-out log loss ---
0.11150094165425457

--- Logistic regression | first five probability/label pairs ---
[[9.33224725e-01 1.00000000e+00]
 [8.74398079e-01 0.00000000e+00]
 [4.97679571e-04 0.00000000e+00]
 [9.99266460e-01 1.00000000e+00]
 [5.19481871e-03 0.00000000e+00]]


## 3. Perceptron: update only on mistakes


In [5]:
negative = rng.normal(loc=[-2.0, -2.0], scale=0.45, size=(120, 2))
positive = rng.normal(loc=[2.0, 2.0], scale=0.45, size=(120, 2))
X = np.vstack([negative, positive])
y = np.array(
    [-1] * len(negative) + [1] * len(positive)
)  # This perceptron contract uses {-1, +1} labels.
X_train, X_test, y_train, y_test = split(X, y)

perceptron = PerceptronClassifier(learning_rate=1.0, max_epochs=50).fit(X_train, y_train)
prediction = perceptron.predict(X_test)

show("Perceptron | mistakes per epoch", perceptron.mistakes_per_epoch_)
show("Perceptron | held-out accuracy", np.mean(prediction == y_test))
show("Perceptron | learned coefficient/intercept", (perceptron.coef_, perceptron.intercept_))


--- Perceptron | mistakes per epoch ---
[1, 0]

--- Perceptron | held-out accuracy ---
1.0

--- Perceptron | learned coefficient/intercept ---
(array([2.18624956, 0.87427842]), -1.0)


## 4. Retrieval checks


In [6]:
assert test_mse < 0.2
assert least_squares_mse < 0.2
assert accuracy > 0.9
assert np.mean(prediction == y_test) == 1.0
assert linear.loss_history_[-1] < linear.loss_history_[0]

show("Algorithm checks | status", "all held-out and convergence assertions passed")


--- Algorithm checks | status ---
all held-out and convergence assertions passed
